# 08 — DPO ciblé sur les erreurs de triage

## 1. Hypothèse et statut

Le DPO généraliste du notebook 03 n'améliorait pas le triage. Le SFT mixte du notebook 07 conserve mieux QA et triage, mais neuf entrées urgentes restent manquées à l'époque 3. **Je n'utilise pas ces neuf entrées de validation pour entraîner le DPO.**

```text
SFT mixte époque 3 → prédictions sur le TRAIN triage (1 888 entrées)
                             ↓ erreurs ESI complètes et cohérentes
               chosen = référence silver / rejected = sortie réelle
                             ↓ séparation par cas, FR/EN groupés
                  DPO train 80 % / contrôle interne 20 %
                             ↓ 32 mises à jour, référence SFT figée
                 validation v3 : QA + triage, aux pas 16 et 32
```

Ce contrôle interne appartient au train SFT : il mesure les préférences DPO, pas la généralisation médicale. Les réponses choisies proviennent d'un consensus de modèles, sans validation clinique. Les différences de style ou de longueur entre référence et sortie peuvent aussi influencer le DPO.

## 2. Expérience fixée avant lancement

| Élément | Choix et raison |
|---|---|
| Départ | Checkpoint 750 du SFT mixte, fin de l'époque 3 ; poids vérifiés |
| Paires | Erreurs réelles du train ; sous-triages **et** sur-triages |
| Exclusions | Sorties invalides, tronquées ou ESI correct ; aucune paire synthétique de remplacement |
| Optimisation | 32 pas, learning rate 0,000002, beta 0,1 ; essai court et conservateur |
| Mémoire | Microbatch 1, accumulation 16, BF16, référence pré-calculée |
| Contrôles | Référence LoRA figée ; prompt et EOS identiques au SFT |
| Mesure | Référence et deux checkpoints réévalués dans la même session |
| Sauvegarde | Poids, paires, prédictions, métriques, MLflow, versions et empreintes sur Drive |

Je ne force pas artificiellement l'équilibre des erreurs : les répartitions par type et langue sont affichées. S'il reste moins de dix cas ou aucun sous-triage critique / sur-triage dans le train DPO, le pilote s'arrête. Une amélioration n'est pas garantie.

## 3. Comment décider

Je fixe ces garde-fous exploratoires avant de voir les résultats : rappel maximal **en hausse**, recul de macro-F1 au plus 1 point, sur-triage en hausse d'au plus 2 points, recul QA d'au plus 3 points sur chacune des deux métriques, sans baisse du taux ESI analysable. J'examine ensuite les cas corrigés et les nouvelles erreurs, en regroupant les traductions.

Ce sont des critères d'expérience, pas des seuils cliniques. Une hausse d'une entrée ne prouve pas un gain robuste ; aucune publication automatique ne suit le calcul. Le test historique reste fermé pour cet essai.

## 4. Exécuter une seule fois sur Colab

Je sélectionne G4 ou A100 80 Go, autorise Drive, puis passe `LANCER_DPO_CIBLE` à `True` dans la cellule suivante. Elle est autonome : je ne relance pas les notebooks précédents. Le dossier `OCR_PROJET_14` sur Drive doit contenir le paquet indiqué et l'expérience `sft_mixte_20261106_091447`.

La cellule génère **1 888 réponses de train + 1 500 réponses de validation**. Elle conserve les résultats sur Drive puis libère la VM après synchronisation. Une erreur d'entraînement est enregistrée dans `statut.json` ; une erreur de sauvegarde empêche la libération.


**Repère de lecture · 08 · Tester des préférences ciblées**

Je teste l’hypothèse suivante : les erreurs réelles du SFT mixte sur le train constituent un signal plus pertinent que des préférences médicales générales. L’expérience est terminée : les deux checkpoints DPO dégradent le triage. Je conserve la référence SFT mixte et documente le résultat négatif.

[← 07 · Préserver QA et triage](07_sft_mixte_colab.ipynb) · [Sommaire du projet](../README.md)

---


In [4]:
LANCER_DPO_CIBLE = False  # Passer à True pour une nouvelle expérience GPU.
if LANCER_DPO_CIBLE:
    import hashlib
    import json
    import os
    from pathlib import Path
    import shutil
    import sqlite3
    import subprocess
    import sys
    import time
    import traceback
    import zipfile
    from datetime import datetime, timezone
    from google.colab import drive, runtime

    REVISION = 'ea980cb0a6c2ae4b936e82123acc929f1cec04c1'
    SHA_ADAPTATEUR = '62d4e5085fd4dd457c5f7b8bbcd0da944ffaf347b8081123f3247dc342db2877'
    ARCHIVE = 'ocr14_dpo_triage_cible_20261106.zip'
    SHA_ARCHIVE = '363a24d0fd58d84144d03e78fb9ad38624f47ec8ab77558d5b4938979bb3a24c'

    def sha256(path):
        with Path(path).open('rb') as flux:
            return hashlib.file_digest(flux, 'sha256').hexdigest()

    def executer(commande, nom):
        """Affiche l'avancement et conserve stdout/stderr, y compris en cas d'échec."""
        print('Étape :', nom, flush=True)
        with (SORTIES / nom).open('a') as log:
            with subprocess.Popen(commande, cwd=RACINE, stdout=subprocess.PIPE,
                    stderr=subprocess.STDOUT, text=True,
                    env={**os.environ, 'PYTHONUNBUFFERED': '1'}) as proc:
                for ligne in proc.stdout:
                    log.write(ligne)
                    log.flush()
                    if 'loss' in ligne or ligne.strip().replace('/', '').isdigit():
                        print(ligne.strip(), flush=True)
                code = proc.wait()
        if code:
            print((SORTIES / nom).read_text()[-6000:])
            raise RuntimeError(f'Échec : {nom}, code {code}')

    if shutil.which('nvidia-smi') is None:
        raise RuntimeError('Sélectionner G4 ou A100 80 Go avant cette cellule.')
    gpu = subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total',
                                   '--format=csv,noheader'], text=True).strip()
    memoire = int(subprocess.check_output(['nvidia-smi', '--query-gpu=memory.total',
                                          '--format=csv,noheader,nounits'], text=True).splitlines()[0])
    if memoire < 60000:
        raise RuntimeError('Cette recette est préparée pour G4 ou A100 80 Go.')
    print('GPU :', gpu)
    drive.mount('/content/drive')
    DRIVE = Path('/content/drive/MyDrive/OCR_PROJET_14')
    paquet = DRIVE / ARCHIVE
    if sha256(paquet) != SHA_ARCHIVE:
        raise ValueError('Archive différente du paquet vérifié.')
    identifiant = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
    RACINE = Path('/content') / f'OCR14_dpo_cible_{identifiant}'
    RACINE.mkdir()
    with zipfile.ZipFile(paquet) as archive:
        for nom in archive.namelist():
            if not (RACINE / nom).resolve().is_relative_to(RACINE):
                raise ValueError('Chemin ZIP invalide.')
        archive.extractall(RACINE)
    SORTIES = DRIVE / 'experiences' / f'dpo_triage_cible_{identifiant}'
    SORTIES.mkdir(parents=True)
    BASE = RACINE / 'mlflow.db'
    statut = {'statut': 'en_cours', 'gpu': gpu, 'test_lu': False,
              'adapter_source_sha256': SHA_ADAPTATEUR, 'archive_sha256': SHA_ARCHIVE}
    debut = time.perf_counter()
    try:
        shutil.copy2(paquet, SORTIES / ARCHIVE)
        executer([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], 'installation.log')
        executer([sys.executable, '-m', 'pip', 'install', '-q',
                  'transformers==5.22.1', 'peft==0.21.2', 'trl==1.17.1', 'datasets==4.7.0',
                  'accelerate==1.15.0', 'mlflow==3.17.2', 'scikit-learn==1.9.1'], 'installation.log')
        import torch
        import mlflow
        if not torch.cuda.is_available() or not torch.cuda.is_bf16_supported():
            raise RuntimeError('CUDA BF16 requis.')
        sys.path.insert(0, str(RACINE))
        from evaluation.comparer_dpo_triage import scores, comparer
        from training.preparer_dpo_triage import lire

        # L'époque 3 a été retenue avant ce DPO. Le modèle actuellement déployé reste inchangé.
        origine = DRIVE / 'experiences/sft_mixte_20261106_091447/entrainement'
        adaptateur = RACINE / 'source/adapter'
        shutil.copytree(origine / 'checkpoints/checkpoint-750', adaptateur,
                        ignore=shutil.ignore_patterns('optimizer.pt', 'scheduler.pt', 'rng_state.pth'))
        if sha256(adaptateur / 'adapter_model.safetensors') != SHA_ADAPTATEUR:
            raise ValueError('Les poids ne sont pas ceux de l’époque 3.')
        shutil.copy2(origine / 'protocole_entrainement.json', adaptateur.parent / 'protocole_entrainement.json')
        shutil.copytree(adaptateur.parent, SORTIES / 'source')
        sources = RACINE / 'data/processed/partitions_v3'
        predictions_train = SORTIES / 'triage_train_minage.jsonl'
        executer([sys.executable, 'evaluation/inference.py', '--phase', 'triage', '--split', 'train',
                  '--data-dir', str(sources), '--adapter', str(adaptateur), '--revision', REVISION,
                  '--dtype', 'bfloat16', '--batch-size', '16', '--stop-token', 'eos',
                  '--max-new-tokens', '192', '--output', str(predictions_train)], 'minage_train.log')
        paires = SORTIES / 'paires'
        executer([sys.executable, 'training/preparer_dpo_triage.py',
                  '--source', str(sources / 'phase2_triage_train.jsonl'),
                  '--predictions', str(predictions_train), '--adapter', str(adaptateur),
                  '--revision', REVISION, '--output', str(paires)], 'preparation_paires.log')
        manifeste = json.loads((paires / 'manifest.json').read_text())
        print('Paires réelles :', json.dumps(manifeste['splits'], ensure_ascii=False, indent=2))
        entrainement = SORTIES / 'entrainement'
        executer([sys.executable, 'training/entrainer_dpo.py', '--adapter', str(adaptateur),
                  '--revision', REVISION, '--data-dir', str(paires), '--output', str(entrainement),
                  '--learning-rate', '2e-6', '--batch-size', '1',
                  '--experiment', 'OCR14-DPO-triage-cible', '--run-name', 'mixte3-erreurs-train-32pas',
                  '--tracking-uri', f'sqlite:///{BASE}',
                  '--artifact-location', (SORTIES / 'artefacts_mlflow').as_uri()], 'entrainement.log')
        statut['entrainement'] = 'termine'
        mlflow.set_tracking_uri(f'sqlite:///{BASE}')
        mlflow.set_experiment('OCR14-DPO-triage-cible')
        bilan = {}
        # Je réévalue la référence dans le même environnement ; pas de comparaison entre moteurs différents.
        candidats = [('sft_mixte_epoque3', adaptateur)] + [
            (f'dpo_pas_{pas}', entrainement / 'checkpoints' / f'checkpoint-{pas}') for pas in (16, 32)]
        for nom, poids in candidats:
            dossier = SORTIES / 'validation' / nom
            dossier.mkdir(parents=True)
            rapports = {}
            with mlflow.start_run(run_name=f'validation-{nom}'):
                mlflow.log_params({'revision': REVISION, 'adapter_sha256': sha256(poids / 'adapter_model.safetensors'),
                                   'split': 'validation', 'gpu': gpu, 'dtype': 'bfloat16', 'batch_size': 16})
                for phase, tokens, prefixe in (('qa', 256, 'phase1_qa'), ('triage', 192, 'phase2_triage')):
                    predictions = dossier / f'{phase}_validation.jsonl'
                    executer([sys.executable, 'evaluation/inference.py', '--phase', phase, '--split', 'validation',
                              '--data-dir', str(sources), '--adapter', str(poids), '--revision', REVISION,
                              '--dtype', 'bfloat16', '--batch-size', '16', '--stop-token', 'eos',
                              '--max-new-tokens', str(tokens), '--output', str(predictions)], f'{nom}_{phase}.log')
                    attendus, obtenus = lire(sources / f'{prefixe}_validation.jsonl'), lire(predictions)
                    if [x['id'] for x in attendus] != [x['id'] for x in obtenus]:
                        raise ValueError('Évaluation incomplète ou désordonnée.')
                    executer([sys.executable, 'evaluation/evaluer.py', str(predictions), '--phase', phase,
                              '--split', 'validation', '--data-dir', str(sources)], f'{nom}_{phase}_scores.log')
                    rapport = predictions.with_name(predictions.stem + '_metriques.json')
                    rapports[phase] = json.loads(rapport.read_text())
                    mlflow.log_artifact(str(rapport))
                    mlflow.log_artifact(str(predictions))
                valeurs = scores(rapports['qa'], rapports['triage'])
                mlflow.log_metrics({k: v for k, v in valeurs.items() if v is not None})
                bilan[nom] = {'scores': valeurs}
                if nom != 'sft_mixte_epoque3':
                    bilan[nom]['comparaison'] = comparer(bilan['sft_mixte_epoque3']['scores'], valeurs)
                (SORTIES / 'comparaison_validation.json').write_text(json.dumps(bilan, ensure_ascii=False, indent=2))
                print(nom, json.dumps(bilan[nom], ensure_ascii=False, indent=2))
        statut['statut'] = 'termine'
    except Exception:
        statut['statut'] = 'echec'
        statut['erreur'] = traceback.format_exc()
        print(statut['erreur'])
    finally:
        # Une erreur de sauvegarde conserve la VM. Je ne libère qu'après copie SQLite et synchronisation Drive.
        statut['duree_secondes'] = round(time.perf_counter() - debut, 1)
        (SORTIES / 'statut.json').write_text(json.dumps(statut, ensure_ascii=False, indent=2))
        if BASE.exists():
            with sqlite3.connect(BASE) as original, sqlite3.connect(SORTIES / 'mlflow.db') as copie:
                original.backup(copie)
                if copie.execute('PRAGMA integrity_check').fetchone()[0] != 'ok':
                    raise RuntimeError('Sauvegarde MLflow invalide ; VM conservée.')
        empreintes = {str(p.relative_to(SORTIES)): sha256(p) for p in SORTIES.rglob('*') if p.is_file()}
        (SORTIES / 'empreintes_resultats.json').write_text(json.dumps(empreintes, indent=2))
        print('Résultats et statut :', SORTIES, statut['statut'])
        drive.flush_and_unmount()
        runtime.unassign()
else:
    print('Expérience non lancée. Lire le protocole puis activer LANCER_DPO_CIBLE pour un run GPU.')


GPU : NVIDIA RTX PRO 6000 Blackwell Server Edition, 97887 MiB
Mounted at /content/drive
Étape : installation.log
Étape : installation.log
Étape : minage_train.log
16/1888
32/1888
48/1888
64/1888
80/1888
96/1888
112/1888
128/1888
144/1888
160/1888
176/1888
192/1888
208/1888
224/1888
240/1888
256/1888
272/1888
288/1888
304/1888
320/1888
336/1888
352/1888
368/1888
384/1888
400/1888
416/1888
432/1888
448/1888
464/1888
480/1888
496/1888
512/1888
528/1888
544/1888
560/1888
576/1888
592/1888
608/1888
624/1888
640/1888
656/1888
672/1888
688/1888
704/1888
720/1888
736/1888
752/1888
768/1888
784/1888
800/1888
816/1888
832/1888
848/1888
864/1888
880/1888
896/1888
912/1888
928/1888
944/1888
960/1888
976/1888
992/1888
1008/1888
1024/1888
1040/1888
1056/1888
1072/1888
1088/1888
1104/1888
1120/1888
1136/1888
1152/1888
1168/1888
1184/1888
1200/1888
1216/1888
1232/1888
1248/1888
1264/1888
1280/1888
1296/1888
1312/1888
1328/1888
1344/1888
1360/1888
1376/1888
1392/1888
1408/1888
1424/1888
1440/1888
1456/

## 5. Lire les sorties avant de conclure

1. `paires/manifest.json` : combien d'erreurs, dans quel sens, dans quelle langue ?
2. `entrainement/resume_entrainement.json` : référence inchangée, tokens contrôlés, loss de préférence ; ce n'est pas le score métier.
3. `comparaison_validation.json` : référence, pas 16 et pas 32 ; garde-fous satisfaits ou non.
4. `validation/` : réponses brutes pour la revue des nouvelles erreurs et des cas corrigés.
5. `mlflow.db` et `artefacts_mlflow/` : paramètres, courbes et preuves à récupérer ensemble.

**Constat actuel :** protocole et code préparés ; aucune mesure du DPO ciblé n'est encore disponible. Le SFT mixte époque 3 reste la référence expérimentale, le SFT v3 reste le modèle du Space.

## 6. Retrouver les choix dans le code

| Script | Responsabilité |
|---|---|
| `evaluation/inference.py --split train` | Minage déterministe, provenance vérifiée, pas de troncature des cas |
| `training/preparer_dpo_triage.py` | Filtrage des erreurs et découpage par cas bilingue |
| `training/preparer_dpo.py` | Format de préférence et EOS natif |
| `training/entrainer_dpo.py` | Optimisation et contrôle de la référence figée |
| `evaluation/comparer_dpo_triage.py` | Garde-fous métier fixés avant le run |

Les scripts exacts sont inclus dans l'archive sauvegardée avec le run. Le [rapport technique](../reports/rapport_technique_ocr14.md) rassemble l'état du projet ; le [README](../README.md) fournit la commande d'ouverture MLflow. Référence de méthode : [documentation officielle TRL DPO](https://huggingface.co/docs/trl/dpo_trainer).


## 6. Résultats du 6 novembre — DPO non retenu

J’ai récupéré le run `dpo_triage_cible_20261106_132004` (G4, 723 s). Les **1 500 prédictions de validation** produisent localement les six mêmes rapports. Les 95 fichiers contrôlés correspondent à leurs SHA-256 ; la base MLflow est intègre. Les poids SFT source, DPO 16/32, états optimiseur, calendrier et RNG sont archivés.

| Validation v3 | SFT mixte réévalué | DPO 16 pas | DPO 32 pas |
|---|---:|---:|---:|
| Accuracy QCM | 56,49 % | 60,31 % | 58,78 % |
| F1 lexical QA | 0,4322 | 0,4371 | 0,4333 |
| Macro-F1 priorité | 0,7875 | 0,7457 | 0,7373 |
| Rappel urgence maximale | 91/100 | 87/100 | 83/100 |
| Accuracy ESI | 74,79 % | 71,01 % | 69,75 % |
| Sur-triage | 18,49 % | 14,71 % | 12,18 % |
| Cohérence bilingue | 78,15 % | 74,79 % | 71,43 % |

**Constat.** Le DPO réduit le sur-triage mais manque davantage d’urgences. Les deux checkpoints échouent aux critères de rappel et de macro-F1 fixés avant l’expérience. La légère amélioration QA ne compense pas ce recul sur l’objectif visé.

**Piste d’explication.** Le minage conserve 440 erreurs sur 1 888 entrées. Après séparation par cas : 355 paires de train (234 sur-triages, 72 sous-triages critiques, 49 autres sous-triages) et 85 de contrôle interne. Les sur-triages représentent 66 % du train DPO. Ce déséquilibre peut orienter l’apprentissage vers une moindre urgence ; il ne démontre pas à lui seul une causalité. Les différences de style chosen/rejected constituent un autre facteur possible.

**Décision.** Je garde le SFT mixte époque 3 comme référence expérimentale ; aucun changement automatique du modèle déployé. Avant un nouvel essai, je ferais relire les préférences et comparerais leur équilibre sur le train, avec une confirmation indépendante. Je ne transforme pas les erreurs de validation en nouvelles paires d’entraînement.

La référence a été réévaluée dans la même session que les deux DPO. Ses petits écarts avec le notebook 07 montrent la sensibilité numérique au contexte d’inférence ; je compare ici les résultats du même run. Le test historique n’a pas été relu. Les 100 entrées urgentes comprennent des traductions corrélées et ne sont pas 100 situations indépendantes.

[Support d’oral et démonstration](https://huggingface.co/spaces/rayakevin/ocr14-triage-fr-en) · [Bilan de vérification](../reports/bilan_dpo_cible_20261106.json)
